In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
import os

# 1. Cấu hình cơ bản
VALID_LABELS = ["Against", "Favor", "None"]
LABEL2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {0: "Against", 1: "Favor", 2: "None"}
MODEL_NAME = "../model/marbert_base" # Vẫn dùng phôi gốc đã tải

# 2. Tiền xử lý (Giữ nguyên)
def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text) 
    text = re.sub(r"@\w+", "", text)             
    text = re.sub(r"\u0640", "", text)           
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text) 
    text = re.sub(r"[إأآا]", "ا", text)        
    text = re.sub(r"ى", "ي", text)              
    text = re.sub(r"ة", "ه", text)              
    text = re.sub(r"(.)\1+", r"\1\1", text)     
    text = text.replace("#", " ")                
    return re.sub(r"\s+", " ", text).strip()

def load_and_prep_data(file_path):
    df = pd.read_csv(file_path, keep_default_na=False)
    df["target"] = df["target"].astype(str).str.strip()
    df["stance"] = df["stance"].astype(str).str.strip()
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    df["input_text"] = df["target"] + " [SEP] " + df["clean_text"]
    df["label"] = df["stance"].map(LABEL2ID)
    return df

print("Đang load dữ liệu...")
train_df = load_and_prep_data("../data/train.csv")
dev_df = load_and_prep_data("../data/dev.csv")

# 3. Tính Class Weights làm nền cho Focal Loss
labels = train_df["label"].tolist()
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    return tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]]).map(tokenize_function, batched=True)
dev_dataset = Dataset.from_pandas(dev_df[["input_text", "label"]]).map(tokenize_function, batched=True)

# 4. ĐỊNH NGHĨA FOCAL LOSS
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight # Tích hợp luôn Class Weights
        self.gamma = gamma   # Hệ số phạt mẫu khó (thường chọn 2.0)

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Custom Trainer sử dụng Focal Loss
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        # Truyền tensor trọng số vào cùng device với model
        loss_fct = FocalLoss(weight=weights_tensor.to(model.device), gamma=2.0)
        loss = loss_fct(outputs.logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f_against = f1_score(labels, predictions, labels=[0], average="macro")
    f_favor = f1_score(labels, predictions, labels=[1], average="macro")
    f_none = f1_score(labels, predictions, labels=[2], average="macro")
    return {"Favg2": (f_favor + f_against) / 2.0, "Favg3": (f_favor + f_against + f_none) / 3.0}

# 5. Huấn luyện Model với Focal Loss
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="../model/focal_loss_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="Favg2",
    greater_is_better=True,
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,      
    processing_class=tokenizer,       
    compute_metrics=compute_metrics,
)

print("Bắt đầu huấn luyện với Focal Loss...")
trainer.train()
trainer.save_model("../model/best_focal_model")
print("Đã lưu mô hình Focal Loss tốt nhất!")

# ====================================================================
# 6. THRESHOLD TUNING (TÌM NGƯỠNG TỐI ƯU CHO FAVG2)
# ====================================================================
print("\n--- Bắt đầu quét Threshold tối ưu trên tập Dev ---")

# Lấy xác suất dự đoán từ model trên tập Dev
predictions = trainer.predict(dev_dataset)
logits = torch.tensor(predictions.predictions)
probs = F.softmax(logits, dim=-1).numpy()
true_labels = dev_df["label"].values

best_favg2 = 0
best_thresholds = {"Against": 0.5, "Favor": 0.5}

# Quét các ngưỡng từ 0.2 đến 0.6 cho Against và Favor
for th_against in np.arange(0.2, 0.6, 0.05):
    for th_favor in np.arange(0.2, 0.6, 0.05):
        custom_preds = []
        for p in probs:
            # Ưu tiên các lớp quyết định Favg2 nếu xác suất vượt ngưỡng
            if p[0] >= th_against:   # Lớp 0: Against
                custom_preds.append(0)
            elif p[1] >= th_favor: # Lớp 1: Favor
                custom_preds.append(1)
            else:
                custom_preds.append(2) # Lớp 2: None
                
        f_against = f1_score(true_labels, custom_preds, labels=[0], average="macro")
        f_favor = f1_score(true_labels, custom_preds, labels=[1], average="macro")
        favg2 = (f_favor + f_against) / 2.0
        
        if favg2 > best_favg2:
            best_favg2 = favg2
            best_thresholds = {"Against": th_against, "Favor": th_favor}

print(f"Điểm Favg2 Mặc định (Argmax): {predictions.metrics['test_Favg2']:.4f}")
print(f"Điểm Favg2 Tối ưu hóa: {best_favg2:.4f}")
print(f"Ngưỡng tốt nhất -> Against: {best_thresholds['Against']:.2f}, Favor: {best_thresholds['Favor']:.2f}")

d:\StanceEval-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang load dữ liệu...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2605.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ../model/marbert_base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bắt đầu huấn luyện với Focal Loss...


Epoch,Training Loss,Validation Loss,Favg2,Favg3
1,No log,0.336704,0.677507,0.561228
2,No log,0.347687,0.784076,0.679580
3,0.351399,0.411602,0.795660,0.696017
4,0.351399,0.475740,0.824715,0.720353


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


Đã lưu mô hình Focal Loss tốt nhất!

--- Bắt đầu quét Threshold tối ưu trên tập Dev ---


Điểm Favg2 Mặc định (Argmax): 0.8247
Điểm Favg2 Tối ưu hóa: 0.8302
Ngưỡng tốt nhất -> Against: 0.55, Favor: 0.30
